# Transformer de snapshot · diario de experimentos (Pablo · A)

**Qué es:** `src/m_nn.py`, una red que mira **todas las tareas de un snapshot a la vez** (self-attention entre tareas)
y da una puntuación a cada una; el softmax sobre la lista dice cuál es la tarea en la que trabaja el controlador.

**Cómo usar este notebook**
1. Ejecuta *Preparación* una vez (carga los datos, ~30 s).
2. Cada bloque de experimentos tiene: **hipótesis** → **variantes** (se cambia UNA cosa respecto a la base) → **tabla**.
3. Escribe la conclusión en la celda *Conclusión* de cada bloque antes de pasar al siguiente.
4. Lo que gane se combina en `MEJOR`, se confirma con 3 semillas y se guarda para el stacking.

**Reglas que respeta el modelo** (`CLAUDE.md` / `EQUIPO.md`)
- Cada snapshot se predice **solo con sus propias filas**: nada de snapshots anteriores o posteriores.
- CV con `Data/folds.csv` (5 folds por controlador; cada sesión queda entera en un fold).
- Early stopping con 3 controladores apartados *dentro* del fold de entrenamiento (nunca mira el fold de validación).
- No usa `sample_id, session, atco, scenario, timestamp, task_name, row_index`.

**Cómo leer las cifras**
- `cv` = top-3 hit rate out-of-fold sobre train+validation. Referencia: LightGBM ≈ 0.655 (con los folds antiguos; pendiente de recalcular con `folds.csv`).
- Las pruebas rápidas usan **1 semilla**: diferencias < ~0.005 son ruido. Lo que parezca ganar se confirma con 3.
- `min_atco` = nota del peor controlador. Nos importa: el privado es **un solo controlador** (023) que no está en train.
- **Decisión a las 21:00:** si la CV queda a más de ~3 puntos del GBM, la red pasa a modelo secundario del ensemble.

> **Resumen (20/10):** mejor versión **`nn_tf_v2` · CV 0.628** (base 0.609), peor controlador 0.566.
> Lo que funciona: **menos** regularización (`ls=0`) y **más** atención entre tareas (`layers=4`), con `lr=5e-4`.
> Lo que no: dropout, ruido, weight decay, 6 capas, lotes pequeños (bloques 1–3) ni las ideas del EDA `rel`, `same_ac`,
> `drop_red` (bloque 9). La red limpia parece cerca de su techo con la información de un solo snapshot.

## Preparación

In [8]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import m_nn
print("device:", m_nn.DEV)

data = m_nn.build_data()          # una sola vez; todas las pruebas lo reutilizan
LGB_CV = 0.655                    # referencia del equipo (actualizar cuando Adri la recalcule con folds.csv)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device: cuda
130 features por tarea, 13175 muestras etiquetadas, 1863 de test, L=59


In [2]:
# Configuración base = la de EQUIPO.md. Cada experimento cambia UNA de estas claves.
BASE = dict(d=64, heads=4, layers=3, drop=0.2, ls=0.1, lr=1e-3, wd=0.05, bs=128, epochs=40, patience=6, noise=0.0,
            rel=False, same_ac=False, drop_red=False)   # las tres últimas: ideas del EDA (bloque 9)


def probar(nombre, motivo, semillas=1, guardar=False, datos=None, **cambios):
    """Lanza la CV con BASE + cambios. Queda apuntado en experiments/nn_runs.csv con su motivo.
    guardar=True genera oof/, preds/ y submissions/ (solo para la versión que va al stacking).
    datos: por defecto `data` (red limpia); en la rama futuro, `data_fut` para la red con futuro."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        if isinstance(v, bool):                       # banderas: --rel, --same-ac, --drop-red
            args += [f"--{k.replace('_', '-')}"] if v else []
        else:
            args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_nn.parse(args)
    a.seeds = semillas
    return m_nn.run(a, data if datos is None else datos)


def lanzar(bloque, variantes, semillas=1, sobre=None, datos=None):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad.
    sobre: cambios que lleva también la base (p. ej. MEJOR); datos: como en probar."""
    sobre = sobre or {}
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas, datos=datos, **sobre))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, datos=datos, **{**sobre, **cambios})))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["Δ vs LGB"] = t.cv - LGB_CV
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}", "Δ vs LGB": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado hasta ahora (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_nn.EXP / "nn_runs.csv")
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "seeds", "d", "layers", "drop", "wd", "noise", "ls", "lr", "bs",
            "best_epochs", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Configuración de `EQUIPO.md` con 3 semillas y `folds.csv`: **CV 0.609** (folds 0.654 / 0.590 / 0.615 / 0.593 / 0.597,
peor controlador 0.543). Queda a ~4.5 puntos del LightGBM.

**Diagnóstico:** las mejores épocas salen casi siempre entre la 3 y la 7 de 40 → el modelo aprende rápido los
controladores de entrenamiento y enseguida empieza a **sobreajustar**. Por eso lo primero es regularizar.

In [3]:
historial()

,when,name,tag,cv,min_atco,seeds,d,layers,drop,wd,noise,ls,lr,bs,best_epochs,secs
0,2026-10-20 17:15,nn_tf,"base EQUIPO.md, folds.csv",0.609,0.5432,3,64,3,0.2,0.05,0.0,0.1,0.001,128,3 7 4 6 6 6 6 6 21 5 3 14 6 6 11,169


## 1 · Regularización

**Hipótesis:** si el problema es sobreajuste, ponerle trabas al modelo debería subir la CV y retrasar la mejor época.

| Cambio | Por qué podría ayudar |
|---|---|
| `drop` 0.3 / 0.4 | apaga neuronas al azar: no puede depender de una sola señal |
| `wd` 0.1 / 0.2 | penaliza pesos grandes: función más suave |
| `noise` 0.05 / 0.1 | ruido en las features al entrenar: no se fía de valores exactos (las etiquetas ya tienen ruido) |

~1 min por variante.

In [4]:
lanzar("reg", [
    ("drop=0.3",   "más dropout contra sobreajuste",        dict(drop=0.3)),
    ("drop=0.4",   "aún más dropout",                       dict(drop=0.4)),
    ("wd=0.1",     "más weight decay",                      dict(wd=0.1)),
    ("wd=0.2",     "aún más weight decay",                  dict(wd=0.2)),
    ("noise=0.05", "ruido en features, robustez",           dict(noise=0.05)),
    ("noise=0.1",  "más ruido en features",                 dict(noise=0.1)),
])


=== reg/base referencia  device=cuda  seeds=1 d=64 heads=4 layers=3 drop=0.2 ls=0.1 lr=0.001 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True
  fold 0 seed 0: best_ep= 3  top3=0.6287  (9s)
fold 0: top3=0.6287
  fold 1 seed 0: best_ep= 6  top3=0.5818  (18s)
fold 1: top3=0.5818
  fold 2 seed 0: best_ep= 6  top3=0.6092  (27s)
fold 2: top3=0.6092
  fold 3 seed 0: best_ep= 5  top3=0.5808  (37s)
fold 3: top3=0.5808
  fold 4 seed 0: best_ep= 6  top3=0.5928  (46s)
fold 4: top3=0.5928
reg/base: CV top3 = 0.5984   (min atco 0.541, media por atco 0.606)   épocas óptimas [3, 6, 6, 5, 6]

=== reg/drop=0.3 más dropout contra sobreajuste  device=cuda  seeds=1 d=64 heads=4 layers=3 drop=0.3 ls=0.1 lr=0.001 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True
  fold 0 seed 0: best_ep=10  top3=0.6246  (14s)
fold 0: top3=0.6246
  fold 1 seed 0: best_ep= 7  top3=0.5767  (24s)
fold 1: top3=0.5767
  fold 2 seed 0: best_ep= 6  top3=0.6033  (35s)
fold 2: top3=0.6033
  

variante,por qué,cv,Δ vs base,Δ vs LGB,veredicto
wd=0.1,más weight decay,0.6011,+0.0027,-0.0539,ruido
base,referencia,0.5984,+0.0000,-0.0566,—
wd=0.2,aún más weight decay,0.5981,-0.0003,-0.0569,ruido
noise=0.05,"ruido en features, robustez",0.5957,-0.0027,-0.0593,ruido
noise=0.1,más ruido en features,0.5896,-0.0088,-0.0654,empeora
drop=0.3,más dropout contra sobreajuste,0.5872,-0.0112,-0.0678,empeora
drop=0.4,aún más dropout,0.5844,-0.0140,-0.0706,empeora


**Conclusión bloque 1 · hipótesis descartada.** Ninguna forma de regularización mejora; dropout y ruido empeoran.

| Variante | CV | Δ vs base (0.598) | |
|---|---|---|---|
| wd=0.1 | 0.601 | +0.003 | ruido |
| wd=0.2 | 0.598 | 0.000 | ruido |
| noise=0.05 | 0.596 | −0.003 | ruido |
| noise=0.1 | 0.590 | −0.009 | empeora |
| drop=0.3 | 0.587 | −0.011 | empeora |
| drop=0.4 | 0.584 | −0.014 | empeora |

- La mejor época **no se retrasa** al regularizar: sigue en la 3–6.
- Por tanto que pare pronto **no es sobreajuste clásico**: aprende enseguida el patrón general y lo que gana después
  son particularidades de los controladores de entrenamiento que no sirven para controladores nuevos. Eso no se
  arregla con dropout.
- La base repetida en los tres bloques da 0.598 / 0.598 / 0.596: el ruido entre ejecuciones iguales es de milésimas,
  pero cada fold varía mucho entre configuraciones, así que < ~0.005 no se considera señal.

## 2 · Capacidad del modelo

**Hipótesis:** con ~13 000 snapshots, un modelo más pequeño puede generalizar mejor a controladores nuevos;
uno más grande solo ayuda si el bloque 1 ha controlado el sobreajuste.

| Cambio | Por qué |
|---|---|
| `d` 32 | menos capacidad para memorizar |
| `d` 128 | más capacidad (por si la base se queda corta) |
| `layers` 2 | menos pasos de atención entre tareas |
| `layers` 4 | más interacción entre tareas (comparar entre aviones, tipos…) |

In [5]:
lanzar("cap", [
    ("d=32",     "modelo más pequeño",         dict(d=32)),
    ("d=128",    "modelo más grande",          dict(d=128)),
    ("layers=2", "menos capas de atención",    dict(layers=2)),
    ("layers=4", "más capas de atención",      dict(layers=4)),
])


=== cap/base referencia  device=cuda  seeds=1 d=64 heads=4 layers=3 drop=0.2 ls=0.1 lr=0.001 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True
  fold 0 seed 0: best_ep= 3  top3=0.6299  (8s)
fold 0: top3=0.6299
  fold 1 seed 0: best_ep= 6  top3=0.5836  (17s)
fold 1: top3=0.5836
  fold 2 seed 0: best_ep= 6  top3=0.6092  (27s)
fold 2: top3=0.6092
  fold 3 seed 0: best_ep= 5  top3=0.5776  (37s)
fold 3: top3=0.5776
  fold 4 seed 0: best_ep= 6  top3=0.5921  (48s)
fold 4: top3=0.5921
cap/base: CV top3 = 0.5983   (min atco 0.538, media por atco 0.607)   épocas óptimas [3, 6, 6, 5, 6]

=== cap/d=32 modelo más pequeño  device=cuda  seeds=1 d=32 heads=4 layers=3 drop=0.2 ls=0.1 lr=0.001 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True
  fold 0 seed 0: best_ep=14  top3=0.6545  (17s)
fold 0: top3=0.6545
  fold 1 seed 0: best_ep= 8  top3=0.5890  (27s)
fold 1: top3=0.5890
  fold 2 seed 0: best_ep=23  top3=0.6187  (52s)
fold 2: top3=0.6187
  fold 3 seed 0: b

variante,por qué,cv,Δ vs base,Δ vs LGB,veredicto
layers=4,más capas de atención,0.6106,+0.0124,-0.0444,mejora
d=128,modelo más grande,0.6056,+0.0074,-0.0494,mejora
d=32,modelo más pequeño,0.6014,+0.0031,-0.0536,ruido
layers=2,menos capas de atención,0.5990,+0.0008,-0.0560,ruido
base,referencia,0.5983,+0.0000,-0.0567,—


**Conclusión bloque 2 · más atención entre tareas ayuda.**

| Variante | CV | Δ vs base (0.598) | |
|---|---|---|---|
| layers=4 | 0.611 | +0.012 | **mejora** (y mejor peor controlador: 0.549) |
| d=128 | 0.606 | +0.007 | mejora justa |
| d=32 | 0.601 | +0.003 | ruido |
| layers=2 | 0.599 | +0.001 | ruido |

- Más capas = más rondas de comparación entre las tareas del snapshot, que es justo lo que el LightGBM no hace
  (puntúa cada tarea por separado). Es la ventaja propia de la red.
- Más anchura (`d`) aporta menos que más profundidad. Se prueba `layers=6` en el bloque 3b.

## 3 · Optimización y pérdida

| Cambio | Por qué |
|---|---|
| `lr` 5e-4 | pasos más pequeños: aprende más despacio y más estable |
| `lr` 2e-3 | por si la base se queda corta |
| `ls` 0.05 / 0.2 | label smoothing: las etiquetas tienen ruido y los controladores no coinciden mucho → suavizar puede ayudar |
| `bs` 64 / 256 | lotes pequeños meten más ruido en el gradiente (regulariza); grandes, más estables |

In [6]:
lanzar("opt", [
    ("lr=5e-4", "aprendizaje más lento",      dict(lr=5e-4)),
    ("lr=2e-3", "aprendizaje más rápido",     dict(lr=2e-3)),
    ("ls=0.05", "menos label smoothing",      dict(ls=0.05)),
    ("ls=0.2",  "más label smoothing",        dict(ls=0.2)),
    ("bs=64",   "lotes pequeños",             dict(bs=64)),
    ("bs=256",  "lotes grandes",              dict(bs=256)),
])


=== opt/base referencia  device=cuda  seeds=1 d=64 heads=4 layers=3 drop=0.2 ls=0.1 lr=0.001 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True
  fold 0 seed 0: best_ep= 3  top3=0.6303  (8s)
fold 0: top3=0.6303
  fold 1 seed 0: best_ep= 6  top3=0.5825  (18s)
fold 1: top3=0.5825
  fold 2 seed 0: best_ep= 6  top3=0.6070  (29s)
fold 2: top3=0.6070
  fold 3 seed 0: best_ep= 5  top3=0.5723  (38s)
fold 3: top3=0.5723
  fold 4 seed 0: best_ep= 6  top3=0.5870  (49s)
fold 4: top3=0.5870
opt/base: CV top3 = 0.5956   (min atco 0.537, media por atco 0.604)   épocas óptimas [3, 6, 6, 5, 6]

=== opt/lr=5e-4 aprendizaje más lento  device=cuda  seeds=1 d=64 heads=4 layers=3 drop=0.2 ls=0.1 lr=0.0005 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True
  fold 0 seed 0: best_ep=10  top3=0.6669  (15s)
fold 0: top3=0.6669
  fold 1 seed 0: best_ep=11  top3=0.5872  (28s)
fold 1: top3=0.5872
  fold 2 seed 0: best_ep= 9  top3=0.6095  (42s)
fold 2: top3=0.6095
  fold 3 se

variante,por qué,cv,Δ vs base,Δ vs LGB,veredicto
ls=0.05,menos label smoothing,0.6165,+0.0209,-0.0385,mejora
lr=5e-4,aprendizaje más lento,0.6088,+0.0132,-0.0462,mejora
bs=256,lotes grandes,0.6011,+0.0055,-0.0539,mejora
lr=2e-3,aprendizaje más rápido,0.5997,+0.0041,-0.0553,ruido
base,referencia,0.5956,+0.0000,-0.0594,—
bs=64,lotes pequeños,0.5921,-0.0035,-0.0629,ruido
ls=0.2,más label smoothing,0.5857,-0.0099,-0.0693,empeora


**Conclusión bloque 3 · el mayor salto viene de bajar el label smoothing.**

| Variante | CV | Δ vs base (0.596) | |
|---|---|---|---|
| ls=0.05 | **0.617** | **+0.021** | **mejora** (la mejor prueba de todas) |
| lr=5e-4 | 0.609 | +0.013 | **mejora** |
| bs=256 | 0.601 | +0.006 | ruido |
| lr=2e-3 | 0.600 | +0.004 | ruido |
| bs=64 | 0.592 | −0.004 | empeora (peor controlador 0.502) |
| ls=0.2 | 0.586 | −0.010 | empeora |

- Menos smoothing mejora y más empeora → se prueba `ls=0` en el bloque 3b.
- Con `lr=5e-4` la mejor época pasa a la 8–11: entrenamiento más estable.
- **Patrón de los tres bloques:** lo que ayuda es *menos* regularización y *más* capacidad de comparar tareas.


## 3b · Seguir la dirección que funciona

**Hipótesis:** si bajar el smoothing y subir capas ayuda, quizá llevarlo más lejos ayude más.

| Cambio | Por qué |
|---|---|
| `ls` 0 | sin label smoothing (0.05 fue lo mejor; 0.1 → 0.2 empeoraba) |
| `layers` 6 | más rondas de atención (4 fue mejor que 3) |


In [3]:

lanzar("dir", [
    ("ls=0",     "sin label smoothing",        dict(ls=0.0)),
    ("layers=6", "aún más capas de atención",  dict(layers=6)),
])



=== dir/base referencia  device=cuda  seeds=1 d=64 heads=4 layers=3 drop=0.2 ls=0.1 lr=0.001 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True


  fold 0 seed 0: best_ep= 3  top3=0.6295  (8s)


fold 0: top3=0.6295


  fold 1 seed 0: best_ep= 6  top3=0.5825  (16s)


fold 1: top3=0.5825


...

fold 3: top3=0.5541


  fold 4 seed 0: best_ep=13  top3=0.5972  (88s)


fold 4: top3=0.5972


dir/layers=6: CV top3 = 0.6041   (min atco 0.523, media por atco 0.610)   épocas óptimas [4, 6, 13, 4, 13]


variante,por qué,cv,Δ vs base,Δ vs LGB,veredicto
ls=0,sin label smoothing,0.6178,+0.0208,-0.0372,mejora
layers=6,aún más capas de atención,0.6041,+0.0071,-0.0509,mejora
base,referencia,0.5970,+0.0000,-0.0580,—


**Conclusión bloque 3b.**

| Variante | CV | Δ vs base (0.597) | peor controlador | |
|---|---|---|---|---|
| ls=0 | **0.618** | **+0.021** | **0.566** | **mejora** |
| layers=6 | 0.604 | +0.007 | 0.523 | mejora justa, peor que layers=4 |

- `ls=0` empata con `ls=0.05` en CV pero sube mucho el peor controlador (0.566 vs 0.535): nos importa porque el
  privado es un solo controlador → se elige **ls=0**.
- 6 capas no mejoran a 4 y empeoran el peor controlador → se queda **layers=4**.

## 4 · Combinar lo que ha funcionado

Pon en `MEJOR` solo los cambios con veredicto **mejora** en los bloques anteriores.
Se comprueba con **3 semillas** frente a la base con 3 semillas (0.609), porque varios cambios que ayudan por separado
no siempre suman.

In [9]:
MEJOR = dict(
    ls=0.0,      # bloques 3 y 3b: +0.021 y mejor peor controlador
    layers=4,    # bloque 2: +0.012
    lr=5e-4,     # bloque 3: +0.013, entrenamiento más estable
)

In [ ]:
# Comprobación con 3 semillas (~4 min). Se puede saltar: el bloque 5 solo necesita la celda de MEJOR.
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   vs base 3 semillas 0.609   vs LGB {LGB_CV}")


=== comb/ls=0.0,layers=4,lr=0.0005 combinación de ganadores  device=cuda  seeds=3 d=64 heads=4 layers=4 drop=0.2 ls=0.0 lr=0.0005 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=True


  fold 0 seed 0: best_ep=15  top3=0.6577  (20s)


  fold 0 seed 1: best_ep=15  top3=0.6488  (40s)


  fold 0 seed 2: best_ep= 7  top3=0.6391  (52s)


fold 0: top3=0.6621


...

  fold 4 seed 1: best_ep= 8  top3=0.6034  (231s)


  fold 4 seed 2: best_ep= 7  top3=0.5983  (243s)


fold 4: top3=0.6224


comb/ls=0.0,layers=4,lr=0.0005: CV top3 = 0.6269   (min atco 0.567, media por atco 0.632)   épocas óptimas [15, 15, 7, 11, 14, 10, 12, 8, 18, 8, 16, 7, 12, 8, 7]


combinación: 0.6269   vs base 3 semillas 0.609   vs LGB 0.655


**Conclusión bloque 4 · los cambios suman.** Con 3 semillas, misma comparación que la base:

| | CV | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 | peor controlador |
|---|---|---|---|---|---|---|---|
| base (EQUIPO.md) | 0.609 | 0.654 | 0.590 | 0.615 | 0.593 | 0.597 | 0.543 |
| **MEJOR** | **0.627** | 0.662 | 0.606 | 0.643 | 0.602 | 0.622 | **0.567** |
| Δ | **+0.018** | +0.008 | +0.016 | +0.028 | +0.009 | +0.025 | +0.024 |

- Mejora en **los 5 folds**, no es suerte de un fold.
- La mejor época pasa de 3–7 a 7–18: con lr más bajo y sin smoothing el entrenamiento es más estable.
- Frente al LightGBM (0.655, con los folds antiguos): **−2.8 puntos**, dentro del margen de ~3 de la decisión de las 21:00.
  Pendiente de confirmar cuando Adri recalcule su CV con `Data/folds.csv`.

## 5 · Versión para el stacking

Si la combinación mejora a la base, se genera la versión definitiva con 3 semillas: guarda `oof/`, `preds/` y
`submissions/` en el formato común. Juanma la usa en el stacking (**nadie más sube a Kaggle**).
Cambia el nombre (`nn_tf_v2`, `v3`…) para no pisar versiones anteriores.

In [4]:
# Necesita BASE, data y MEJOR (celdas de Preparación y de MEJOR). ~4 min, sobrescribe oof/preds/submissions de NOMBRE.
NOMBRE = "nn_tf_v2"
probar(NOMBRE, "versión para stacking: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)


=== nn_tf_v2 versión para stacking: {'ls': 0.0, 'layers': 4, 'lr': 0.0005}  device=cuda  seeds=3 d=64 heads=4 layers=4 drop=0.2 ls=0.0 lr=0.0005 wd=0.05 bs=128 epochs=40 patience=6 noise=0.0 n_es=3 folds= quick=False


  fold 0 seed 0: best_ep=15  top3=0.6585  (19s)


  fold 0 seed 1: best_ep=12  top3=0.6404  (35s)


  fold 0 seed 2: best_ep= 7  top3=0.6395  (46s)


fold 0: top3=0.6673


...

  test seed 0: 11 épocas (261s)


  test seed 1: 11 épocas (276s)


  test seed 2: 11 épocas (292s)


Guardado oof/nn_tf_v2.csv, preds/nn_tf_v2.csv, submissions/sub_nn_tf_v2.csv
Comprobar: python Data/scorer.py --labelled Data/validation.csv oof/sub_nn_tf_v2_validation.csv


0.6279316888045541

In [5]:
# Comprobación con el scorer oficial sobre las muestras de validation.csv
!cd {ROOT} && {sys.executable} Data/scorer.py --labelled Data/validation.csv oof/sub_{NOMBRE}_validation.csv

top-3 hit rate: 0.6342  (1252 samples)


**Resultado de la versión para el stacking (`nn_tf_v2`)**

| | CV top-3 | peor controlador | validation.csv (scorer oficial) |
|---|---|---|---|
| `nn_tf` (base EQUIPO.md) | 0.609 | 0.543 | 0.597 |
| **`nn_tf_v2`** (`ls=0, layers=4, lr=5e-4`) | **0.628** | **0.566** | **0.634** |

- Reproduce la cifra del bloque 4 (0.627 → 0.628): el resultado es estable.
- Salidas en formato común, listas para Juanma: `oof/nn_tf_v2.csv`, `preds/nn_tf_v2.csv`, `submissions/sub_nn_tf_v2.csv`.
- A ~2.7 puntos del LightGBM (0.655, folds antiguos) → por ahora la red **entra en el ensemble**. Lo que decide de verdad es
  si la mezcla con el LightGBM sube la CV (bloque 6).


## 6 · Análisis de errores

**Pregunta:** ¿dónde falla la red? Sirve para decidir qué features pedir a Pau/Adri y qué atacar en la red.
Se mira sobre el OOF (predicciones que la red hizo sin haber visto a ese controlador). `analizar_errores` vale para
cualquier OOF del equipo (`"lgb_rank"`, `"types"`…), así se pueden comparar los fallos de varios modelos.


In [2]:

TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]


def analizar_errores(modelo="nn_tf_v2"):
    cols = ["sample_id", "row_index", "target_index", "task_index", "atco", "seq_len"]
    d = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=cols) for f in ("train", "validation")])
    d = d.merge(pd.read_csv(ROOT / "oof" / f"{modelo}.csv"), on=["sample_id", "row_index"])
    d["rk"] = d.groupby("sample_id").score.rank(ascending=False, method="first")
    d["tipo"] = d.task_index.map(dict(enumerate(TIPOS)))
    t = d[d.target_index == 1].assign(hit=lambda x: x.rk <= 3)
    fallos = t[~t.hit]
    print(f"{modelo}: top-3 {t.hit.mean():.3f}   top-1 {(t.rk == 1).mean():.3f}")

    por_tipo = t.groupby("tipo").agg(muestras=("hit", "size"), acierto_top3=("hit", "mean"))
    por_tipo["% de los fallos"] = fallos.tipo.value_counts(normalize=True).reindex(por_tipo.index).fillna(0)
    display(por_tipo.sort_values("muestras", ascending=False).style.format(
        {"acierto_top3": "{:.2f}", "% de los fallos": "{:.0%}"}).background_gradient(subset=["acierto_top3"], cmap="RdYlGn"))

    t["longitud"] = pd.cut(t.seq_len, [0, 25, 35, 45, 60], labels=["≤25", "26–35", "36–45", "46–59"])
    display(t.groupby("longitud", observed=True).hit.agg(muestras="size", acierto_top3="mean").round(3).T)

    primero = d[(d.rk == 1) & d.sample_id.isin(fallos.sample_id)][["sample_id", "tipo"]].rename(columns={"tipo": "red_pone_1º"})
    conf = fallos[["sample_id", "tipo"]].rename(columns={"tipo": "correcto"}).merge(primero, on="sample_id")
    print("Pares más frecuentes en los fallos (tipo correcto -> tipo que la red pone 1º):")
    display(conf.groupby(["correcto", "red_pone_1º"]).size().sort_values(ascending=False).head(8).rename("fallos").to_frame())
    print(f"Posición del correcto cuando falla: mediana {fallos.rk.median():.0f}, "
          f"en 4º-5º el {fallos.rk.le(5).mean():.0%} de los fallos")
    return t


errores = analizar_errores("nn_tf_v2")


nn_tf_v2: top-3 0.628   top-1 0.338


,muestras,acierto_top3,% de los fallos
tipo,,,
transfer,3527,0.84,12%
assume,3366,0.80,14%
zone,2121,0.75,11%
entry_coord,1068,0.29,15%
return_to_route,937,0.37,12%
exit_coord,745,0.05,14%
exit_conditions,448,0.13,8%
conflict,390,0.33,5%
qos,257,0.17,4%


longitud,≤25,26–35,36–45,46–59
muestras,1918.000,4003.000,5204.000,2050.000
acierto_top3,0.737,0.629,0.616,0.554


Pares más frecuentes en los fallos (tipo correcto -> tipo que la red pone 1º):


,,fallos
correcto,red_pone_1º,
exit_coord,transfer,304
return_to_route,transfer,293
entry_coord,assume,288
assume,transfer,265
entry_coord,transfer,237
exit_coord,assume,226
transfer,assume,195
zone,assume,180


Posición del correcto cuando falla: mediana 7, en 4º-5º el 34% de los fallos



**Conclusión del análisis de errores (`nn_tf_v2`):** la red acierta bien los tres tipos frecuentes y casi nunca los raros,
que son la mayoría de los fallos.

| Tipo correcto | Muestras | Acierto top-3 | % de todos los fallos |
|---|---|---|---|
| transfer | 3.527 | **0.84** | 12 % |
| assume | 3.366 | **0.80** | 14 % |
| zone_conflict | 2.121 | **0.76** | 11 % |
| entry_coordination | 1.068 | 0.29 | **15 %** |
| exit_coordination | 745 | **0.05** | **14 %** |
| return_to_route | 937 | 0.37 | **12 %** |
| exit_conditions | 448 | 0.13 | 8 % |
| conflict_resolution | 390 | 0.33 | 5 % |

1. **Juega a lo seguro:** en los fallos pone primero transfer, assume o zone el 90 % de las veces. Confusiones típicas:
   exit_coord → transfer (304), return_to_route → transfer (293), entry_coord → assume (288).
2. **exit_coordination casi nunca la acierta (5 %)**: es donde más margen hay.
3. **entry_coordination vs assume** del mismo avión: mismo avión y mismas features, el caso difícil conocido.
4. **Cuanto más larga la lista, peor:** 0.74 con ≤25 tareas, 0.55 con más de 45.
5. Un tercio de los fallos son por poco (correcto en 4ª–5ª posición); en el resto está lejos (mediana 7ª).
6. Top-1 solo 0.34: el top-3 rescata mucho.

**Qué pedir a Pau/Adri:** features que digan *cuándo toca coordinar*, sobre todo para exit_coordination:
`k_outCoordMismatch`, `k_coordinationToolActivation`, `k_out_NOTIFIED` cruzadas con el tipo; para return_to_route, la
desviación de ruta (`non_conformance_agreed_route_deviation_horizontally`). Las columnas existen pero la señal llega débil.

**Por qué el futuro ayuda tanto (LightGBM 0.655 → 0.83):** desde un solo snapshot es muy difícil saber *cuándo* el
controlador se pone con una coordinación; su siguiente instrucción al avión lo delata. Es justo el tipo de fallo que domina aquí.


## 7 · Siguientes ideas (no implementadas todavía)

Por orden de prioridad:
- **Features nuevas para exit_coordination y return_to_route** (propuestas en la sección 6): **Pau está trabajando en ellas**.
  Cuando estén en `features.py`, probarlas aquí como un bloque más (y en la red con futuro).

- **Mezclar rankings red + LightGBM** en la CV: si la mezcla sube la CV del LightGBM, la red aporta al ensemble aunque sola sea peor. Necesita `oof/lgb_*.csv` de Adri con `Data/folds.csv`.
- **Usar el OOF del LightGBM de Adri como feature de entrada** (variante 3 de `EQUIPO.md`). Necesita `oof/lgb_*.csv` y
  `preds/lgb_*.csv` con los mismos folds.
- ~~**Sesgo de atención "mismo avión"**~~ → implementado como `same_ac`, se prueba en el bloque 9. Antes:: que la atención sepa qué tareas son del mismo avión (sobre todo assume vs
  entry_coordination, `l_twin_AE`). El avión sale de `task_name`, que no se puede usar como feature: consultarlo con el equipo.
- **Si se permiten otros snapshots:** que la red vea también la lista anterior y la siguiente.

## Historial completo

In [6]:
historial()

,when,name,tag,cv,min_atco,seeds,d,layers,drop,wd,noise,ls,lr,bs,best_epochs,secs
0,2026-10-20 18:00,nn_tf_v2,"versión para stacking: {'ls': 0.0, 'layers': 4...",0.6279,0.5664,3,64,4,0.2,0.05,0.00,0.00,0.0005,128,15 12 7 11 14 10 12 8 19 8 16 7 12 8 7,291
1,2026-10-20 17:54,"comb/ls=0.0,layers=4,lr=0.0005",combinación de ganadores,0.6269,0.5674,3,64,4,0.2,0.05,0.00,0.00,0.0005,128,15 15 7 11 14 10 12 8 18 8 16 7 12 8 7,243
2,2026-10-20 17:48,dir/ls=0,sin label smoothing,0.6178,0.5662,1,64,3,0.2,0.05,0.00,0.00,0.0010,128,7 8 10 8 7,52
3,2026-10-20 17:40,opt/ls=0.05,menos label smoothing,0.6165,0.5347,1,64,3,0.2,0.05,0.00,0.05,0.0010,128,6 16 18 7 6,68
4,2026-10-20 17:35,cap/layers=4,más capas de atención,0.6106,0.5490,1,64,4,0.2,0.05,0.00,0.10,0.0010,128,4 17 9 8 8,78
5,2026-10-20 17:15,nn_tf,"base EQUIPO.md, folds.csv",0.6090,0.5432,3,64,3,0.2,0.05,0.00,0.10,0.0010,128,3 7 4 6 6 6 6 6 21 5 3 14 6 6 11,169
6,2026-10-20 17:38,opt/lr=5e-4,aprendizaje más lento,0.6088,0.5381,1,64,3,0.2,0.05,0.00,0.10,0.0005,128,10 11 9 8 8,67
7,2026-10-20 17:33,cap/d=128,modelo más grande,0.6056,0.5335,1,128,3,0.2,0.05,0.00,0.10,0.0010,128,5 6 5 6 4,52
8,2026-10-20 17:49,dir/layers=6,aún más capas de atención,0.6041,0.5225,1,64,6,0.2,0.05,0.00,0.10,0.0010,128,4 6 13 4 13,88
9,2026-10-20 17:32,cap/d=32,modelo más pequeño,0.6014,0.5441,1,32,3,0.2,0.05,0.00,0.10,0.0010,128,14 8 23 5 29,91



## 9 · Ideas del EDA de Pau (`notebooks/01_eda.ipynb`)

Ideas sacadas del EDA y del análisis de errores (sección 6), adaptadas a la red. Todas usan **solo filas del mismo
snapshot**. Se prueban de una en una **sobre `MEJOR`** (`ls=0, layers=4, lr=5e-4`), con 1 semilla.

| Opción | Qué hace | De dónde sale / por qué podría ayudar |
|---|---|---|
| `rel` | z-score de 8 columnas clave (`time_to_sector`, \|`time_to_sector`\|, `x_task_age`, `x_state_age`, `a_since_flight`, `k_timeToExitPoint`, `distance_to_zone`, `time_to_MD`) **dentro del snapshot** y **dentro de snapshot × tipo** | EDA, conclusión 1: importa la posición *relativa* dentro de la muestra, no el valor absoluto. Es la idea 5 de Pau (FIFO por tipo) generalizada. Debería ayudar en listas largas (0.55 con >45 tareas) |
| `same_ac` | sesgo de atención aprendido entre tareas del **mismo avión** | EDA, caso gemelo assume/entry (idea 2 de Pau); entry_coordination es el 15 % de los fallos. El avión sale de `task_name` solo para agrupar, como en `hierarchical/m_types.py` |
| `drop_red` | quita una columna de cada par con \|ρ\| > 0.95 (11 columnas y sus indicadores de vacío) | EDA, sección 6: "para modelos lineales/NN conviene podar". Menos ruido de entrada |

No se prueba el PCA: el propio EDA concluye que no rankea mejor que las `l_rank_*`.
**Lo que mejore aquí se aplica también a la red con futuro** (rama `futuro`).

~1–1,5 min por variante.


In [ ]:
lanzar("eda", [
    ("rel",      "features relativas dentro del snapshot y del tipo",  dict(rel=True)),
    ("same_ac",  "atención sabe qué tareas son del mismo avión",       dict(same_ac=True)),
    ("drop_red", "quitar columnas casi redundantes",                    dict(drop_red=True)),
], sobre=MEJOR)

**Conclusión bloque 9 · ninguna idea del EDA mejora la red.**

| Variante | CV | Δ vs base (0.615) | peor controlador | |
|---|---|---|---|---|
| drop_red | 0.617 | +0.002 | 0.546 | ruido |
| rel | 0.613 | −0.002 | 0.550 | ruido |
| same_ac | 0.610 | −0.005 | 0.544 | ruido / empeora un poco |

- **`rel`**: la atención ya compara las tareas del snapshot entre sí, y las `l_rank_*` originales ya dan la posición dentro
  del tipo. Darle los z-scores hechos no añade información. Cambia qué folds acierta (fold 3 +0.024, fold 2 −0.020),
  pero en total queda igual.
- **`same_ac`**: las tareas del mismo avión ya se reconocen solas porque comparten todas las columnas del avión;
  decírselo explícitamente no ayuda en el caso gemelo.
- **`drop_red`**: a la red no le molestan las columnas duplicadas.
- **Lectura general:** a la red limpia no le falta "forma" de mirar los datos, le falta **información**. Con lo que hay en
  un solo snapshot parece estar cerca de su techo (~0.62–0.63). Las ideas del EDA están pensadas para árboles, que sí
  necesitan las comparaciones hechas a mano: tiene más sentido que las pruebe Adri en el LightGBM.
- **9b no se lanza:** no hay nada que combinar; `nn_tf_v2` sigue siendo la versión para el stacking.

### 9b · Combinación: no se lanza

Ninguna de las tres ideas mejora, así que no hay nada que combinar. **`nn_tf_v2` (CV 0.628) sigue siendo la versión de la red
para el stacking.** Las opciones `--rel`, `--same-ac` y `--drop-red` se quedan en `src/m_nn.py`, desactivadas por defecto, por si
se quieren reprobar con otras features. Tampoco se prueban en la red con futuro (rama `futuro`): si no ayudan a la limpia,
no hay motivo para esperar otra cosa.